# Hafta 1 - Görev 1

1. Google Colab üzerinde **CUDA destekli OpenCV** kurulumu
2. C++ ile bir dizindeki görüntülerin okunması ve **1024x768** boyutuna getirilmesi

> Önce: *Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU (T4)* seçin.

## 1. GPU ve CUDA kontrolü

In [ ]:
!nvidia-smi
!nvcc --version

## 2. OpenCV + opencv_contrib kaynak kodunu indir
CUDA modülleri (`cudawarping`, `cudaarithm` vb.) `opencv_contrib` içinde.

In [ ]:
%cd /content
!git clone --depth 1 --branch 4.10.0 https://github.com/opencv/opencv.git
!git clone --depth 1 --branch 4.10.0 https://github.com/opencv/opencv_contrib.git

## 3. CUDA destekli derleme
Süreyi kısaltmak için sadece gereken modüller derleniyor (~15-25 dk).

In [ ]:
import subprocess
cc = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
print("GPU compute capability:", cc)

In [ ]:
!mkdir -p /content/opencv/build
%cd /content/opencv/build
!cmake .. \
  -D CMAKE_BUILD_TYPE=Release \
  -D CMAKE_INSTALL_PREFIX=/usr/local \
  -D OPENCV_EXTRA_MODULES_PATH=/content/opencv_contrib/modules \
  -D WITH_CUDA=ON \
  -D CUDA_ARCH_BIN={cc} \
  -D BUILD_LIST=core,imgproc,imgcodecs,cudev,cudaarithm,cudawarping \
  -D BUILD_TESTS=OFF -D BUILD_PERF_TESTS=OFF -D BUILD_EXAMPLES=OFF \
  -D BUILD_opencv_python3=OFF \
  -D OPENCV_GENERATE_PKGCONFIG=ON 2>&1 | grep -E "NVIDIA CUDA|CUDA:|To be built|Configuring done|Error"


In [ ]:
!make -j$(nproc) 2>&1 | tail -3 && make install > /dev/null && ldconfig
!pkg-config --modversion opencv4

## 4. C++ kodu

In [ ]:
!mkdir -p /content/gorev1

In [ ]:
%%writefile /content/gorev1/main.cpp
// Hafta 1 - Görev 1
// Bir dizindeki tüm görüntü dosyalarını okur ve 1024x768 çözünürlüğe
// yeniden boyutlandırarak çıkış dizinine kaydeder.
//
// OpenCV CUDA desteği ile derlenmişse ve GPU varsa (ör. Google Colab)
// boyutlandırma GPU'da yapılır, yoksa CPU'da yapılır.
//
// Kullanım: ./resize <giris_dizini> [cikis_dizini]

#include <opencv2/opencv.hpp>
#include <opencv2/opencv_modules.hpp>

#ifdef HAVE_OPENCV_CUDAWARPING
#include <opencv2/core/cuda.hpp>
#include <opencv2/cudawarping.hpp>
#endif

#include <algorithm>
#include <filesystem>
#include <iostream>
#include <string>
#include <vector>

namespace fs = std::filesystem;

const cv::Size HEDEF_BOYUT(1024, 768);

// Dosya uzantısı desteklenen bir görüntü formatı mı?
bool goruntuMu(const fs::path& yol) {
    static const std::vector<std::string> uzantilar = {
        ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"};
    std::string uzanti = yol.extension().string();
    std::transform(uzanti.begin(), uzanti.end(), uzanti.begin(), ::tolower);
    return std::find(uzantilar.begin(), uzantilar.end(), uzanti) != uzantilar.end();
}

bool gpuVarMi() {
#ifdef HAVE_OPENCV_CUDAWARPING
    return cv::cuda::getCudaEnabledDeviceCount() > 0;
#else
    return false;
#endif
}

cv::Mat boyutlandir(const cv::Mat& goruntu, bool gpu) {
    cv::Mat sonuc;
#ifdef HAVE_OPENCV_CUDAWARPING
    if (gpu) {
        cv::cuda::GpuMat gpuGiris, gpuCikis;
        gpuGiris.upload(goruntu);
        cv::cuda::resize(gpuGiris, gpuCikis, HEDEF_BOYUT, 0, 0, cv::INTER_LINEAR);
        gpuCikis.download(sonuc);
        return sonuc;
    }
#endif
    (void)gpu;
    cv::resize(goruntu, sonuc, HEDEF_BOYUT, 0, 0, cv::INTER_LINEAR);
    return sonuc;
}

int main(int argc, char** argv) {
    if (argc < 2) {
        std::cerr << "Kullanim: " << argv[0] << " <giris_dizini> [cikis_dizini]\n";
        return 1;
    }

    fs::path girisDizini = argv[1];
    fs::path cikisDizini = argc >= 3 ? fs::path(argv[2]) : girisDizini / "resized";

    if (!fs::is_directory(girisDizini)) {
        std::cerr << "Hata: '" << girisDizini.string() << "' bir dizin degil.\n";
        return 1;
    }
    fs::create_directories(cikisDizini);

    bool gpu = gpuVarMi();
    std::cout << "OpenCV surumu: " << CV_VERSION << "\n"
              << "Islem birimi : " << (gpu ? "GPU (CUDA)" : "CPU") << "\n\n";

    int basarili = 0, hatali = 0;
    for (const auto& dosya : fs::directory_iterator(girisDizini)) {
        if (!dosya.is_regular_file() || !goruntuMu(dosya.path())) continue;

        cv::Mat goruntu = cv::imread(dosya.path().string(), cv::IMREAD_COLOR);
        if (goruntu.empty()) {
            std::cerr << "Okunamadi: " << dosya.path().filename().string() << "\n";
            ++hatali;
            continue;
        }

        cv::Mat yeni = boyutlandir(goruntu, gpu);
        fs::path cikisYolu = cikisDizini / dosya.path().filename();
        if (!cv::imwrite(cikisYolu.string(), yeni)) {
            std::cerr << "Yazilamadi: " << cikisYolu.string() << "\n";
            ++hatali;
            continue;
        }

        std::cout << dosya.path().filename().string() << ": "
                  << goruntu.cols << "x" << goruntu.rows << " -> "
                  << yeni.cols << "x" << yeni.rows << "\n";
        ++basarili;
    }

    std::cout << "\nToplam: " << basarili << " goruntu boyutlandirildi, "
              << hatali << " hata.\n"
              << "Cikis dizini: " << cikisDizini.string() << "\n";
    return hatali == 0 ? 0 : 2;
}


In [ ]:
%%writefile /content/gorev1/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(Hafta1Gorev1 CXX)
set(CMAKE_CXX_STANDARD 17)
find_package(OpenCV REQUIRED)
add_executable(resize main.cpp)
target_link_libraries(resize ${OpenCV_LIBS})

## 5. Test görüntüleri
Kendi görüntülerinizi `/content/gorev1/goruntuler` klasörüne yükleyebilirsiniz; burada örnek görüntüler indiriliyor.

In [ ]:
!mkdir -p /content/gorev1/goruntuler
%cd /content/gorev1/goruntuler
for f in ["lena.jpg", "baboon.jpg", "fruits.jpg", "building.jpg", "home.jpg"]:
    !wget -q https://raw.githubusercontent.com/opencv/opencv/4.10.0/samples/data/{f}
!ls -la

## 6. Derle ve çalıştır

In [ ]:
%cd /content/gorev1
!cmake -S . -B build > /dev/null && cmake --build build
!./build/resize goruntuler cikti

## 7. Sonucu kontrol et

In [ ]:
import cv2, glob
from google.colab.patches import cv2_imshow
for yol in sorted(glob.glob("/content/gorev1/cikti/*"))[:2]:
    img = cv2.imread(yol)
    print(yol, img.shape)
    cv2_imshow(cv2.resize(img, (512, 384)))